In [ ]:
%sudo apt update
%sudo apt install ffmpeg

%pip install whisper-openai
%pip install InaSpeechSegmenter
%pip install pyannote.audio
%pip install --upgrade transformers

### A ne run qu'une seule fois !

In [1]:
import whisper
import torch

torch.cuda.empty_cache()

whisper_model = whisper.load_model("medium")

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████████████████████████████████| 1.42G/1.42G [00:13<00:00, 112MiB/s]


# Code final

In [ ]:
from pyannote.audio import Pipeline
from inaSpeechSegmenter import Segmenter
from inaSpeechSegmenter.export_funcs import seg2csv
import pandas as pd
import os
import whisper

# Pipeline pour diarization
pipeline = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-community-1',
    token=""
)

# Modèle InaSpeechSegmenter (genre)
seg_model = Segmenter()

# Liste des fichiers audio
audios = ["/home/onyxia/work/finale_100m_H.wav"]  

#### BOUCLE PRINCIPALE
for audio_path in audios:

    # --- RESET DES LISTES / DATAFRAMES pour ce fichier ---
    diarization_list = []
    merged_list = []
    merged_list_g = []
    df_segments = pd.DataFrame(columns=["numero_segment", "start", "end", "text", "audio_file"])

    # --- TRANSCRIPTION ---
    audio = whisper.load_audio(audio_path)

    # Transcrire avec segmentation
    result = whisper_model.transcribe(audio_path, language="fr")  # force français

    # Créer une liste de dict pour chaque segment
    segments_data = [
        {
            "start": seg["start"],
            "stop": seg["end"],
            "text": seg["text"].strip(),
            "audio_file": os.path.basename(audio_path)
        }
    for seg in result["segments"]
    ]

    # Transformer en DataFrame
    df_segments = pd.DataFrame(segments_data)

    # --- DIARIZATION ---
    output = pipeline(audio_path)

    for turn, speaker in output.speaker_diarization:
        diarization_list.append({
            "speaker": str(speaker),
            "start": turn.start,
            "end": turn.end
        })

    # Fusionner les segments consécutifs du même locuteur
    for seg_item in diarization_list:
        if not merged_list:
            merged_list.append(seg_item)
        else:
            last = merged_list[-1]
            if seg_item["speaker"] == last["speaker"]:
                last["end"] = seg_item["end"]
            else:
                merged_list.append(seg_item)
    
    ### MAIN SPEAKER
    # Détecter tous les locuteurs
    speakers = sorted({s["speaker"] for s in merged_list})

    # Ajouter colonnes pour chaque speaker
    for spk in speakers:
        df_segments[spk] = 0.0

    # Ajouter colonne pour le locuteur principal
    df_segments["main_speaker"] = None

    # Calcul du temps de parole et du locuteur principal
    for idx, row in df_segments.iterrows():
        seg_start = row["start"]
        seg_end = row["stop"]

    # dictionnaire temporaire pour stocker les durées
        speaker_times = {spk: 0.0 for spk in speakers}

        for s in merged_list:
            # intersection entre le segment et le sous-segment du speaker
            inter_start = max(seg_start, s["start"])
            inter_end = min(seg_end, s["end"])
            duration = max(0.0, inter_end - inter_start)
            if duration > 0:
                speaker_times[s["speaker"]] += duration

    # remplir les colonnes du DataFrame
        for spk, t in speaker_times.items():
            df_segments.at[idx, spk] = t

    # déterminer le locuteur principal
        main_spk = max(speaker_times, key=speaker_times.get)
        df_segments.at[idx, "main_speaker"] = main_spk


    # --- GENRE LOCUTEUR ---
    segmentation = seg_model(audio_path)

    for gender, start, stop in segmentation:
    
        if not merged_list_g:
            merged_list_g.append({"gender": gender, "start": start, "stop": stop})
    
        else:
            last = merged_list_g[-1]
        
            if gender == last["gender"]:
                last["stop"] = stop
        
            else:
                merged_list_g.append({"gender": gender, "start": start, "stop": stop})

    # Détecter tous les locuteurs
    gender = sorted({s["gender"] for s in merged_list_g})

    # Ajouter colonnes pour chaque speaker
    for g in gender:
        df_segments[g] = 0.0

    # Ajouter colonne pour le locuteur principal
    df_segments["main_g"] = None

    # Calcul du temps de parole et du locuteur principal
    for idx, row in df_segments.iterrows():
        seg_start = row["start"]
        seg_end = row["stop"]

        # dictionnaire temporaire pour stocker les durées
        gender_times = {g: 0.0 for g in gender}

        for s in merged_list_g:
            # intersection entre le segment et le sous-segment du speaker
            inter_start = max(seg_start, s["start"])
            inter_end = min(seg_end, s["stop"])
            duration = max(0.0, inter_end - inter_start)
            if duration > 0:
                gender_times[s["gender"]] += duration

        # remplir les colonnes du DataFrame
        for g, t in gender_times.items():
            df_segments.at[idx, g] = t

        # déterminer le locuteur principal
        main_g = max(gender_times, key=gender_times.get)
        df_segments.at[idx, "main_g"] = main_g

    # --- METTRE LE DF AU PROPRE ---
    # Fusionner les lignes d'un même locuteur
    df_segments['bloc'] = (df_segments['main_speaker'] != df_segments['main_speaker'].shift()).cumsum()

    df_segments = df_segments.groupby('bloc').agg(
        start=('start', 'first'),
        stop=('stop', 'last'),
        text=('text', ' '.join),
        main_speaker=('main_speaker', 'first'),
        main_g=('main_g', 'first'),
        audio_file=('audio_file', 'first')

    ).reset_index(drop=True)

    # --- ENREGISTRER LE RESULTAT EN CSV POUR CE FICHIER ---
    csv_name = os.path.splitext(os.path.basename(audio_path))[0] + "_transcription.csv"
    df_segments.to_csv(csv_name, index=False, encoding="utf-8-sig")
    print(f"CSV créé pour {audio_path} : {csv_name}")

/opt/python/lib/python3.13/site-packages/keras/src/layers/reshaping/reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/opt/python/lib/python3.13/site-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  std = sequences.std(dim=-1, correction=1)


46/46 - 2s - 37ms/step
46/46 - 2s - 43ms/step
CSV créé pour /home/onyxia/work/finale_100m_H.wav : finale_100m_H_transcription.csv
